# 2D Segmentation with the Datamint Trainer API

This notebook shows how to train a **DeepLab V3+ semantic segmentation** model on the **BUSI** (Breast Ultrasound Images) dataset using Datamint's **Trainer API** — a high-level wrapper that handles dataset loading, model creation, training, experiment tracking, and model registration from a small amount of code.

## Comparison with the manual Workflow

The conventional way requires you to:
1. Define transforms, loss function, metrics, and a full LightningModule (~150 lines)
2. Configure MLflow logger, callbacks, and Trainer (~50 lines)
3. Build and wire together `DatamintDataModule`, `L.Trainer`, etc.

With the Trainer API, **all of that is replaced by ~3 lines**:

```python
from datamint.lightning.trainers import DeepLabV3PlusTrainer

trainer = DeepLabV3PlusTrainer(project='MyProject')
results = trainer.fit()
```

## What You'll Learn

1. Upload data to Datamint
2. Train with `DeepLabV3PlusTrainer` using **zero configuration**
3. Train with `SemanticSegmentation2DTrainer` while swapping in a custom external model
4. Visualise predictions
5. Register and deploy the trained model

## Required Dependencies

```bash
pip install datamint
```

In [10]:
from datamint import Api

PROJECT_NAME = "deeplabv3plus_Segmentation_Tutorial"
api = Api()

## 1. Setup: Create Project

Create (or retrieve) a Datamint project for this tutorial.

In [11]:
proj = api.projects.create(
    name=PROJECT_NAME,
    description="Tutorial project for DeepLabV3+ segmentation on BUSI dataset",
    exists_ok=True # Just return the existing project if it already exists
)
proj

HTTPStatusError: Client error '403 Forbidden' for url 'https://api.datamint.io/projects'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

## 2. Dataset Preparation: Download and Upload BUSI

This section downloads the BUSI dataset and uploads it to Datamint.
If you already have the data inside Datamint, you can **skip to Section 3**.

In this section, we will:
- Download the BUSI dataset
- Upload ultrasound images to Datamint
- Upload corresponding segmentation masks
- Create train/val/test splits

### 2.1 Download BUSI Dataset

The BUSI dataset is available from Kaggle. For this tutorial, we'll use the breast ultrasound images dataset.

> Al-Dhabyani W, Gomaa M, Khaled H, Fahmy A. Dataset of breast ultrasound images. Data in Brief. 2020 Feb;28:104863. DOI: 10.1016/j.dib.2019.104863.

In [ ]:
import os
import requests
import zipfile
from pathlib import Path

BUSI_URL = "https://www.kaggle.com/api/v1/datasets/download/sabahesaraki/breast-ultrasound-images-dataset"
DATA_DIR = Path("/tmp/BUSI_dataset")

if not DATA_DIR.exists():
    print("Downloading BUSI dataset...")
    response = requests.get(BUSI_URL, stream=True)
    response.raise_for_status()
    zip_path = DATA_DIR / "Dataset_BUSI.zip"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with open(zip_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    os.remove(zip_path)
    print("Download complete!")
else:
    print(f"Dataset already exists at {DATA_DIR}")

### 2.2 Find image and mask paths

In [ ]:
base_dir = DATA_DIR / "Dataset_BUSI_with_GT"
classes = ["benign", "malignant", "normal"]

image_paths = []
label_paths = []

for cls in classes:
    cls_dir = base_dir / cls
    cls_images = sorted([p for p in cls_dir.glob("*.png") if "_mask" not in p.name])
    for img_p in cls_images:
        mask_p = cls_dir / f"{img_p.stem}_mask.png"
        if mask_p.exists():
            image_paths.append(img_p)
            label_paths.append(mask_p)

print(f"Found {len(image_paths)} images and {len(label_paths)} masks")

### 2.3 Upload images and masks to Datamint

In [ ]:
# Upload images
uploaded_resources = api.resources.upload_resources(
    [str(p) for p in image_paths],
    tags=['busi', 'ultrasound', 'breast'],
    publish_to=proj,
    progress_bar=True,
)
print(f"Uploaded {len(uploaded_resources)} images")

In [ ]:
from tqdm.auto import tqdm

# Get resources from project
all_resources = list(api.resources.get_list(project_name=PROJECT_NAME, tags=['busi']))
filename_to_resource = {r.filename: r for r in all_resources}

# Upload segmentation masks
for img_path, label_path in tqdm(zip(image_paths, label_paths), total=len(image_paths)):
    if 'normal' in img_path.parent.name:
        continue  # Normal images have no lesion masks
    resource = filename_to_resource[img_path.name]
    cls_name = img_path.parent.name  # 'benign' or 'malignant'

    api.annotations.upload_segmentations(
        resource=resource,
        file_path=label_path,
        name=cls_name,
        imported_from="Original GT BUSI Dataset",
    )

print("Segmentation masks uploaded successfully!")

### 2.4 Tag train/val/test splits

We split the dataset into three subsets using tags for reproducibility.

| Split | Percentage | Purpose |
|-------|------------|---------|
| Train | 70% | Model training |
| Validation | 15% | Hyperparameter tuning, early stopping |
| Test | 15% | Final model evaluation |

In [ ]:
import random

all_resources = list(api.resources.get_list(project_name=PROJECT_NAME, tags=['busi']))
all_resources.sort(key=lambda r: r.filename)

random.seed(42)
random.shuffle(all_resources)

n_total = len(all_resources)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)

train_resources = all_resources[:n_train]
val_resources = all_resources[n_train:n_train + n_val]
test_resources = all_resources[n_train + n_val:]

api.projects.assign_splits(proj, train_resources, split_name='train')
api.projects.assign_splits(proj, val_resources, split_name='val')
api.projects.assign_splits(proj, test_resources, split_name='test')

print(f"Train: {len(train_resources)}, Val: {len(val_resources)}, Test: {len(test_resources)}")

## 3. Training with the Trainer API

`DeepLabV3PlusTrainer` wraps the full training pipeline in a single object. It automatically:

1. Builds an `ImageDataset` from the project (or a sliced 2D view if the project contains 3D volumes)
2. Applies standard augmentations — flips, brightness/contrast, ImageNet normalisation
3. Instantiates a **DeepLab v3+** model with an ImageNet-pretrained encoder
4. Sets up **BCE + Dice loss**, **Mean IoU** and **Generalised Dice** metrics
5. Configures MLflow logging, early stopping, and model checkpointing
6. Trains, validates, and evaluates on the test split

The key architecture-specific parameter is `decoder_atrous_rates`, which controls the dilation rates of the **ASPP** (Atrous Spatial Pyramid Pooling) module — DeepLab v3+'s mechanism for capturing multi-scale context. The default `(12, 24, 36)` works well for most medical imaging tasks.

In [ ]:
from datamint.lightning import DeepLabV3PlusTrainer


trainer = DeepLabV3PlusTrainer(
    project=PROJECT_NAME,
    image_size=256,
    batch_size=16,
    max_epochs=12,
    accelerator='auto',
    # register_model_name='MyModelName' # Default is project name
)

In [ ]:
results = trainer.fit()

That's it!

Let's inspect the results:

In [ ]:
# Inspect the results dictionary
print("Test results:")
for metric_dict in results['test_results']:
    for k, v in metric_dict.items():
        print(f"  {k}: {v:.4f}")

## 4. Visualize predictions

Let's visualize the model predictions against ground truth on test samples.

In [ ]:
import torch
import numpy as np
from matplotlib import pyplot as plt
from datamint.utils.visualization import show, draw_masks
from torchmetrics.functional.segmentation import mean_iou

model = trainer.model
model.eval()

# Access the test dataset from the internal datamodule
test_dataset = trainer.datamodule.test_dataloader().dataset
class_names = trainer.datamodule.dataset.seglabel_list

fig, axes = plt.subplots(3, 2, figsize=(12, 18))

for row in range(3):
    idx = np.random.choice(len(test_dataset))
    sample = test_dataset[idx]
    image = sample['image']         # (C, H, W)
    mask_gt = sample['segmentations']  # (#classes+1, H, W) — includes background

    with torch.inference_mode():
        logits = model(image.unsqueeze(0).to(model.device))
        mask_pred = (logits[0] > 0).cpu()

    # Ground truth (skip background channel)
    gt_overlay = draw_masks(image, mask_gt[1:], alpha=0.5)
    axes[row, 0].set_title(f"Ground Truth (sample {idx})")
    show(gt_overlay, ax=axes[row, 0])

    # Prediction
    pred_overlay = draw_masks(image, mask_pred, alpha=0.5)
    axes[row, 1].set_title("Prediction")
    show(pred_overlay, ax=axes[row, 1])

    iou = mean_iou(
        mask_pred.unsqueeze(0).long(),
        mask_gt[1:].unsqueeze(0).long(),
        num_classes=len(class_names),
        input_format='one-hot',
    )
    print(f"Sample {idx} — IoU: {iou.max():.1%} — Classes: {class_names}")

plt.tight_layout()
plt.show()

## 6. Deployment

The trainer **automatically** created and registered a deployment adapter in MLflow.
You can deploy it directly to the Datamint server:

In [ ]:
job = api.deploy.start(
    model_name=PROJECT_NAME,
    model_alias="latest",
    with_gpu=False,
)

print(f"Deployment job started!")
print(f"Job ID: {job.id}")
print(f"Status: {job.status}")

In [ ]:
# Check deployment status
job = api.deploy.get_by_id(job.id)

print(f"Job Status: {job.status}")
print(f"Progress: {job.progress_percentage}%")

if job.error_message:
    print(f"Error: {job.error_message}")

### 6.1 Remote inference

In [ ]:
from datamint import Api
PROJECT_NAME = "deeplabv3plus_Segmentation_Tutorial"
api = Api()

In [ ]:
r = api.resources.get_list(project_name=PROJECT_NAME, limit=1)[0]

In [ ]:
inf_job = api.inference.submit(
    model_name=PROJECT_NAME,
    model_alias="latest",
    resource_id=r.id
)
inf_job.wait() # Wait for inference to complete

visualize predictions:

In [ ]:
from matplotlib import pyplot as plt

# plot all predictions using matplotlib
preds = inf_job.predictions[0]
plt.figure(figsize=(6, 6))
plt.imshow(preds[0].mask, cmap='gray')
plt.title("Predicted Mask")
plt.axis('off')

In [ ]:
# Open the Datamint dashboard for this project
proj.show()